# CellFM frozen embeddings for synthetic bulk RNA-seq patients

> **Cross-domain application:** CellFM was pretrained on single-cell RNA-seq. This notebook treats each bulk RNA-seq patient as one independent model input for a reproducible, frozen-embedding smoke test. It does not claim performance, train a classifier, or fine-tune CellFM.

The protocol follows AACR 2026 Abstract 5478, *Benchmarking gene expression foundation models on bulk RNA-Seq data*: use each bulk sample as one input, retain the model's preprocessing, freeze the pretrained model, and extract fixed representations. Because a model-specific bulk pooling rule is not published, this notebook exports both representations exposed by the implementation:

- **CLS embedding:** encoder output at token 0.
- **Mean embedding:** arithmetic mean over valid gene-token outputs only; the CLS token, two ST special tokens, and padding are excluded.

Synthetic raw integer counts are used solely to validate the pipeline. Replace the synthetic table later with a patient-by-gene raw-count table while preserving the mapping and preprocessing cells.

In [27]:
from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
from pathlib import Path

SEED = 7
N_PATIENTS = 3
N_MAPPED_INPUT_GENES = 3_200
N_UNMAPPED_INPUT_GENES = 100
BATCH_SIZE = 1

# Find this notebook reliably even after upstream CellFM changes cwd.
cwd = Path.cwd().resolve()
candidate_dirs = [cwd, *cwd.parents, cwd / 'CellFM', cwd.parent / 'CellFM']
CELLFM_DIR = next(
    (p for p in candidate_dirs if (p / 'patient_embedding_smoke_test.ipynb').is_file()),
    None,
)
if CELLFM_DIR is None:
    raise FileNotFoundError(f'Cannot locate the CellFM notebook from {cwd}')

VENDOR_DIR = CELLFM_DIR / 'vendor'
if VENDOR_DIR.is_dir() and str(VENDOR_DIR) not in sys.path:
    sys.path.insert(0, str(VENDOR_DIR))
if sys.version_info[:2] != (3, 9):
    raise RuntimeError(
        f'Use the Python 3.9 notebook kernel; current: {sys.version.split()[0]}'
    )

import anndata as ad
import mindspore
import numpy as np
import pandas as pd
import scipy
import torch

CELLFM_TORCH_DIR = Path(
    os.environ.get('CELLFM_TORCH_DIR', CELLFM_DIR / 'CellFM-torch')
).expanduser().resolve()
CHECKPOINT_PATH = Path(
    os.environ.get('CELLFM_CKPT', CELLFM_DIR / 'weights' / 'CellFM_80M_weight.ckpt')
).expanduser().resolve()
OUTPUT_DIR = CELLFM_DIR / 'bulk_smoke_outputs'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'CellFM root:  {CELLFM_DIR}')
print(f'CellFM-torch: {CELLFM_TORCH_DIR}')
print(f'Checkpoint:   {CHECKPOINT_PATH}')
print(f'Device:       {DEVICE}')

CellFM root:  /data01/private/projects/omiclass/CellFM
CellFM-torch: /data01/private/projects/omiclass/CellFM/CellFM-torch
Checkpoint:   /data01/private/projects/omiclass/CellFM/weights/CellFM_80M_weight.ckpt
Device:       cuda


In [28]:
required_files = [
    CELLFM_TORCH_DIR / 'model.py',
    CELLFM_TORCH_DIR / 'layers' / 'utils.py',
    CELLFM_TORCH_DIR / 'csv' / 'expand_gene_info.csv',
    CELLFM_TORCH_DIR / 'csv' / 'updated_hgcn.tsv',
]
missing = [str(p) for p in required_files if not p.is_file()]
if missing:
    raise FileNotFoundError('Incomplete CellFM-torch checkout:\n- ' + '\n- '.join(missing))
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'CellFM checkpoint not found: {CHECKPOINT_PATH}')

# Upstream utilities read csv/... relative to their repository root.
os.chdir(CELLFM_TORCH_DIR)
if str(CELLFM_TORCH_DIR) not in sys.path:
    sys.path.insert(0, str(CELLFM_TORCH_DIR))

from layers.utils import Config_80M, Prepare, build_dataset
from model import Cell_FM

## 1. Create three synthetic raw bulk-count profiles

Rows are independent bulk patients and columns are gene identifiers. The table deliberately contains more than 2,048 mapped, expressed genes plus unmapped identifiers so vocabulary coverage and sequence truncation are exercised. Counts remain raw non-negative integers at this stage.

In [29]:
rng = np.random.default_rng(SEED)
patient_ids = pd.Index(
    [f'bulk_patient_{i + 1:02d}' for i in range(N_PATIENTS)],
    name='patient_id',
)

gene_info = pd.read_csv(
    CELLFM_TORCH_DIR / 'csv' / 'expand_gene_info.csv', index_col=0
)
vocab_names = gene_info.index.astype(str).to_numpy()
if len(np.unique(vocab_names)) != len(vocab_names):
    raise ValueError('CellFM vocabulary contains duplicate gene identifiers.')
if N_MAPPED_INPUT_GENES > len(vocab_names):
    raise ValueError('Requested more synthetic genes than the CellFM vocabulary contains.')

# Uniform sampling defines the synthetic assay's gene universe only. It does
# not choose the 2,048 genes retained by CellFM; upstream Prepare.sample does that.
mapped_input_genes = rng.choice(
    vocab_names, size=N_MAPPED_INPUT_GENES, replace=False
)
unmapped_input_genes = np.array(
    [f'NOT_IN_CELLFM_{i + 1:04d}' for i in range(N_UNMAPPED_INPUT_GENES)]
)
input_genes = np.concatenate([mapped_input_genes, unmapped_input_genes])

# Gamma-Poisson counts with patient-specific expression programs. Adding one
# makes every mapped gene expressed, guaranteeing that the 2,048-token limit is tested.
base_means = rng.lognormal(mean=3.5, sigma=0.8, size=len(input_genes))
raw_counts = np.empty((N_PATIENTS, len(input_genes)), dtype=np.int64)
for patient_idx in range(N_PATIENTS):
    patient_means = base_means.copy()
    start = patient_idx * 150
    patient_means[start:start + 150] *= 4.0
    rate = rng.gamma(shape=5.0, scale=patient_means / 5.0)
    raw_counts[patient_idx] = rng.poisson(rate).astype(np.int64) + 1

bulk_raw_df = pd.DataFrame(raw_counts, index=patient_ids, columns=input_genes)
assert (bulk_raw_df.to_numpy() >= 0).all()
assert np.issubdtype(bulk_raw_df.to_numpy().dtype, np.integer)
print(f'Raw bulk matrix: {bulk_raw_df.shape}')
display(bulk_raw_df.iloc[:, :8])

Raw bulk matrix: (3, 3300)


,SELENOW,CAPN8,EEF1E1-BLOC1S5,LMX1B,MICALL2-DT,ITGA7,MIR6767,BAG6
patient_id,,,,,,,,
bulk_patient_01,87,126,103,960,234,26,121,685
bulk_patient_02,65,11,28,163,34,15,35,49
bulk_patient_03,72,26,20,151,29,10,26,83


## 2. Exact vocabulary mapping and CellFM preprocessing

Input identifiers are matched exactly to the HGNC-symbol index in `expand_gene_info.csv`; unmapped genes are excluded. The original `read_h5ad` library-size cap is reproduced first: libraries above 100,000 reads are divided by `T / 100000` and rounded.

For inference, training augmentation is disabled (`dw=False`, `mask_ratio=0`). The actual upstream `Prepare.sample` implementation performs sequence selection: when more than 2,048 mapped genes are expressed, it samples without replacement with probability proportional to `log1p(raw count)`, then restores original feature order. Selected expression values are transformed by `log1p(count / mapped_library_size * 10000)`. No arbitrary first-2,048 truncation is used.

In [30]:
vocab_to_id = {gene: idx + 1 for idx, gene in enumerate(vocab_names)}
mapped_mask = bulk_raw_df.columns.isin(vocab_to_id)
mapped_genes = bulk_raw_df.columns[mapped_mask].astype(str).to_numpy()
unmapped_genes = bulk_raw_df.columns[~mapped_mask].astype(str).to_numpy()
mapped_gene_ids = np.array([vocab_to_id[g] for g in mapped_genes], dtype=np.int32)

# Reproduce CellFM-torch read_h5ad's raw-count library cap before mapping.
raw_library_sizes = bulk_raw_df.sum(axis=1).to_numpy(dtype=np.float64)
library_divisor = np.maximum(1.0, raw_library_sizes / 1e5)
capped_all_counts = np.rint(
    bulk_raw_df.to_numpy(dtype=np.float64) / library_divisor[:, None]
).astype(np.int32)
mapped_counts = capped_all_counts[:, mapped_mask]
mapped_library_sizes = mapped_counts.sum(axis=1).astype(np.float32)

class BulkPatientDataset:
    """Expose one complete mapped bulk profile per CellFM input."""

    def __init__(self, counts, gene_ids, ids):
        self.counts = np.asarray(counts, dtype=np.float32)
        self.gene_ids = np.asarray(gene_ids, dtype=np.int32)
        self.patient_ids = pd.Index(ids)
        self.library_sizes = self.counts.sum(axis=1).astype(np.float32)

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        # Final three values are unused labels required by upstream collate_fn.
        return (
            self.counts[idx],
            self.gene_ids,
            self.library_sizes[idx],
            0,
            0,
            0.0,
        )

dataset = BulkPatientDataset(mapped_counts, mapped_gene_ids, patient_ids)
cfg = Config_80M()
# Required upstream defaults omitted by Config_80M itself.
cfg.ecs = True
cfg.ecs_threshold = 0.8
cfg.add_zero = True
cfg.pad_zero = True
cfg.mask_ratio = 0.0
cfg.recompute = False
cfg.device = str(DEVICE)
cfg.ckpt_path = str(CHECKPOINT_PATH)

prep = Prepare(
    pad_len=cfg.nonz_len,
    pad=0,  # PyTorch encoder adds CLS + two ST tokens internally.
    mask_ratio=0.0,
    dw=False,
    random=False,
)
loader = build_dataset(
    dataset, prep=prep, batch_size=BATCH_SIZE, pad_zero=True,
    drop=False, shuffle=False,
)

mapping_overview = {
    'original_number_of_genes': int(bulk_raw_df.shape[1]),
    'mapped_to_cellfm_vocabulary': int(len(mapped_genes)),
    'unmapped_from_cellfm_vocabulary': int(len(unmapped_genes)),
    'cellfm_vocabulary_size': int(len(vocab_names)),
    'model_sequence_length': int(cfg.nonz_len),
}
print(json.dumps(mapping_overview, indent=2))

{
  "original_number_of_genes": 3300,
  "mapped_to_cellfm_vocabulary": 3200,
  "unmapped_from_cellfm_vocabulary": 100,
  "cellfm_vocabulary_size": 27855,
  "model_sequence_length": 2048
}


## 3. Load and freeze pretrained CellFM

The complete model is placed in evaluation mode and every parameter has `requires_grad=False`. Embedding extraction below is additionally wrapped in `torch.inference_mode()`.

In [31]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = Cell_FM(
    n_gene=len(vocab_names),
    cfg=cfg,
    ckpt_path=str(CHECKPOINT_PATH),
    device=str(DEVICE),
).to(DEVICE)
model.load_model(weight=True, moment=False)
model.eval()
model.requires_grad_(False)

total_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
parameter_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
assert trainable_parameters == 0
assert not model.training
if torch.cuda.is_available():
    assert DEVICE.type == 'cuda'
print(f'Model parameters: {total_parameters:,}; trainable: {trainable_parameters:,}')

[Load Report]
Missing keys in PyTorch model: []
Unexpected keys in checkpoint: []
Model parameters: 86,193,666; trainable: 0


/data01/private/projects/omiclass/CellFM/CellFM-torch/model.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(init_scale=1.0)


## 4. Extract CLS and valid-gene mean embeddings twice

`encoded[:, 0, :]` is the CLS representation. CellFM-torch then has two ST special tokens, so gene-token outputs start at position 3. Mean pooling uses `zero_idx` to include only retained, non-padding gene tokens. Resetting NumPy's seed before each pass makes CellFM's documented count-weighted gene selection reproducible.

In [32]:
def extract_frozen_embeddings(selection_seed: int):
    np.random.seed(selection_seed)  # Prepare.sample uses global NumPy RNG.
    cls_batches = []
    mean_batches = []
    selected_gene_ids = []
    retained_counts = []
    grad_flags = []

    with torch.inference_mode():
        for batch in loader:
            grad_flags.append(torch.is_grad_enabled())
            expr = batch['dw_nzdata'].to(DEVICE)
            gene = batch['nonz_gene'].to(DEVICE)
            st_feat = batch['ST_feat'].to(DEVICE)
            valid_mask = batch['zero_idx'].to(DEVICE)
            assert expr.shape[1] == gene.shape[1] == valid_mask.shape[1] == cfg.nonz_len

            encoded, _ = model.net.encode(expr, gene, st_feat, valid_mask)
            cls = encoded[:, 0, :]
            gene_tokens = encoded[:, 3:, :]  # exclude CLS and two ST tokens
            mask = valid_mask.unsqueeze(-1).to(gene_tokens.dtype)
            mean = (gene_tokens * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)

            cls_batches.append(cls.float().cpu().numpy())
            mean_batches.append(mean.float().cpu().numpy())
            for row in range(gene.shape[0]):
                keep = valid_mask[row].bool()
                selected_gene_ids.append(gene[row, keep].cpu().numpy().copy())
                retained_counts.append(int(keep.sum().item()))

    assert not any(grad_flags), 'Gradient tracking was enabled during inference.'
    return (
        np.concatenate(cls_batches),
        np.concatenate(mean_batches),
        selected_gene_ids,
        np.asarray(retained_counts, dtype=np.int32),
    )

cls_embeddings, mean_embeddings, selected_ids, retained = extract_frozen_embeddings(SEED)
cls_repeat, mean_repeat, selected_ids_repeat, retained_repeat = extract_frozen_embeddings(SEED)

np.testing.assert_allclose(cls_embeddings, cls_repeat, rtol=1e-5, atol=1e-6)
np.testing.assert_allclose(mean_embeddings, mean_repeat, rtol=1e-5, atol=1e-6)
np.testing.assert_array_equal(retained, retained_repeat)
assert all(np.array_equal(a, b) for a, b in zip(selected_ids, selected_ids_repeat))
print(f'CLS embeddings:  {cls_embeddings.shape}')
print(f'Mean embeddings: {mean_embeddings.shape}')

CLS embeddings:  (3, 1536)
Mean embeddings: (3, 1536)


## 5. Validate, report vocabulary coverage, and save artifacts

In [33]:
assert cls_embeddings.shape == mean_embeddings.shape == (N_PATIENTS, cfg.enc_dims)
assert np.isfinite(cls_embeddings).all()
assert np.isfinite(mean_embeddings).all()

def pairwise_l2_rows(values):
    return [
        float(np.linalg.norm(values[i] - values[j]))
        for i in range(len(values)) for j in range(i + 1, len(values))
    ]

cls_pairwise_l2 = pairwise_l2_rows(cls_embeddings)
mean_pairwise_l2 = pairwise_l2_rows(mean_embeddings)
cls_embeddings_differ = all(distance > 1e-8 for distance in cls_pairwise_l2)
mean_embeddings_differ = all(distance > 1e-8 for distance in mean_pairwise_l2)
# In this CellFM implementation the exposed CLS token can be invariant across
# inputs. AACR 5478 specifies valid-gene average pooling when no model-specific
# pooling is available, so mean pooling is the required patient representation.
assert mean_embeddings_differ, 'Mean-pooled patient embeddings are not distinct.'
if not cls_embeddings_differ:
    print(
        'Diagnostic: CellFM CLS embeddings are invariant across patients; '
        'use cellfm_mean_embeddings.csv for the bulk representation.'
    )

nonzero_mapped = (mapped_counts > 0).sum(axis=1).astype(int)
discarded_by_limit = nonzero_mapped - retained
assert (nonzero_mapped > cfg.nonz_len).all(), 'Synthetic data did not exercise the limit.'
assert (discarded_by_limit > 0).all()
preprocessing_report = pd.DataFrame(
    {
        'original_genes': bulk_raw_df.shape[1],
        'mapped_to_cellfm': len(mapped_genes),
        'unmapped_to_cellfm': len(unmapped_genes),
        'mapped_nonzero_after_library_cap': nonzero_mapped,
        'retained_for_model_input': retained,
        'discarded_by_sequence_limit': discarded_by_limit,
        'raw_library_size': raw_library_sizes.astype(np.int64),
        'mapped_library_size_after_cap': mapped_library_sizes.astype(np.int64),
    },
    index=patient_ids,
)
display(preprocessing_report)

# Record the exact retained and discarded genes for every patient.
id_to_vocab = {idx + 1: gene for idx, gene in enumerate(vocab_names)}
selection_records = []
for row, patient_id in enumerate(patient_ids):
    selected_set = set(map(int, selected_ids[row]))
    nonzero_ids = mapped_gene_ids[mapped_counts[row] > 0]
    for gene_id in nonzero_ids:
        gene_id = int(gene_id)
        selection_records.append(
            {
                'patient_id': patient_id,
                'gene_id': gene_id,
                'gene_symbol': id_to_vocab[gene_id],
                'selection_status': (
                    'retained' if gene_id in selected_set
                    else 'discarded_by_sequence_limit'
                ),
            }
        )
gene_selection_report = pd.DataFrame(selection_records)
selection_counts = gene_selection_report.groupby(
    ['patient_id', 'selection_status']
).size().unstack(fill_value=0)
np.testing.assert_array_equal(selection_counts['retained'].to_numpy(), retained)
np.testing.assert_array_equal(
    selection_counts['discarded_by_sequence_limit'].to_numpy(), discarded_by_limit
)

embedding_columns = [f'emb_{i:04d}' for i in range(cfg.enc_dims)]
cls_df = pd.DataFrame(cls_embeddings, index=patient_ids, columns=embedding_columns)
mean_df = pd.DataFrame(mean_embeddings, index=patient_ids, columns=embedding_columns)
assert cls_df.index.equals(patient_ids) and mean_df.index.equals(patient_ids)

try:
    cellfm_torch_commit = subprocess.check_output(
        ['git', '-C', str(CELLFM_TORCH_DIR), 'rev-parse', 'HEAD'], text=True
    ).strip()
except Exception:
    cellfm_torch_commit = None

summary = {
    'application': 'cross-domain bulk RNA-seq embedding with a single-cell-pretrained CellFM model',
    'scope': 'synthetic frozen-embedding smoke test only; no classification or fine-tuning',
    'reference': {
        'title': 'AACR 2026 Abstract 5478: Benchmarking gene expression foundation models on bulk RNA-Seq data',
        'doi': '10.1158/1538-7445.AM2026-5478',
    },
    'input': {
        'data_type': 'synthetic raw bulk RNA-seq integer gene counts',
        'patients': int(N_PATIENTS),
        'matrix_dimensions_patients_by_genes': list(map(int, bulk_raw_df.shape)),
        **mapping_overview,
        'per_patient_vocabulary_coverage': preprocessing_report.reset_index().to_dict(orient='records'),
    },
    'preprocessing': {
        'identifier_mapping': 'exact match to expand_gene_info.csv HGNC_gene index',
        'raw_library_cap': 'round(count / max(1, total_counts / 1e5)) as in CellFM-torch read_h5ad',
        'sequence_selection': 'upstream Prepare.sample; weighted without replacement by log1p(raw count)',
        'sequence_selection_seed': int(SEED),
        'sequence_length': int(cfg.nonz_len),
        'value_normalization': 'log1p(selected_count / mapped_library_size * 1e4)',
        'mask_ratio': float(cfg.mask_ratio),
        'count_downsampling_enabled': bool(prep.dw),
    },
    'model': {
        'checkpoint_path': str(CHECKPOINT_PATH),
        'checkpoint_size_bytes': int(CHECKPOINT_PATH.stat().st_size),
        'cellfm_torch_commit': cellfm_torch_commit,
        'total_parameters': int(total_parameters),
        'trainable_parameters': int(trainable_parameters),
        'parameter_bytes': int(parameter_bytes),
        'evaluation_mode': bool(not model.training),
    },
    'pooling': {
        'cls': 'encoder token 0',
        'mean': 'arithmetic mean of valid gene-token outputs; excludes CLS, two ST tokens, and padding',
    },
    'output': {
        'rows_per_file': int(N_PATIENTS),
        'embedding_dimension': int(cfg.enc_dims),
        'cls_file': 'cellfm_cls_embeddings.csv',
        'mean_file': 'cellfm_mean_embeddings.csv',
        'gene_selection_file': 'cellfm_gene_selection_report.csv',
    },
    'verification': {
        'all_finite': True,
        'patients_have_distinct_cls_embeddings': bool(cls_embeddings_differ),
        'patients_have_distinct_mean_embeddings': bool(mean_embeddings_differ),
        'recommended_bulk_representation': 'valid-gene mean pooling',
        'repeat_allclose_rtol': 1e-5,
        'repeat_allclose_atol': 1e-6,
        'repeat_selection_identical': True,
        'gradients_calculated': False,
        'cuda_available': bool(torch.cuda.is_available()),
        'device_used': str(DEVICE),
        'cls_pairwise_l2': cls_pairwise_l2,
        'mean_pairwise_l2': mean_pairwise_l2,
    },
    'software_versions': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scipy': scipy.__version__,
        'anndata': ad.__version__,
        'torch': torch.__version__,
        'cuda_runtime': torch.version.cuda,
        'mindspore': mindspore.__version__,
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bulk_raw_df.to_csv(OUTPUT_DIR / 'synthetic_bulk_raw_counts.csv')
preprocessing_report.to_csv(OUTPUT_DIR / 'cellfm_preprocessing_report.csv')
gene_selection_report.to_csv(
    OUTPUT_DIR / 'cellfm_gene_selection_report.csv', index=False
)
cls_df.to_csv(OUTPUT_DIR / 'cellfm_cls_embeddings.csv')
mean_df.to_csv(OUTPUT_DIR / 'cellfm_mean_embeddings.csv')
with (OUTPUT_DIR / 'cellfm_bulk_smoke_summary.json').open('w') as handle:
    json.dump(summary, handle, indent=2)

print(f'Validated and wrote {N_PATIENTS} frozen bulk-patient embeddings to {OUTPUT_DIR}')

Diagnostic: CellFM CLS embeddings are invariant across patients; use cellfm_mean_embeddings.csv for the bulk representation.


,original_genes,mapped_to_cellfm,unmapped_to_cellfm,mapped_nonzero_after_library_cap,retained_for_model_input,discarded_by_sequence_limit,raw_library_size,mapped_library_size_after_cap
patient_id,,,,,,,,
bulk_patient_01,3300,3200,100,3200,2048,1152,172261,96830
bulk_patient_02,3300,3200,100,3200,2048,1152,173889,96754
bulk_patient_03,3300,3200,100,3200,2048,1152,173885,96997


Validated and wrote 3 frozen bulk-patient embeddings to /data01/private/projects/omiclass/CellFM/bulk_smoke_outputs
